# How to Assess Encoding Quality

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/assess_encoding_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fassess_encoding_quality.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/assess_encoding_quality.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


Two complementary tools for evaluating an encoding before training:
- `encoding_sensitivity()` + `score_encoding()` — feature influence and expressibility
- `detect_barren_plateau()` — analytical gradient variance risk estimate

In [ ]:
!pip install -q quprep

In [ ]:
import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning
from quprep.core.dataset import Dataset

print(f"quprep {qd.__version__}")

rng = np.random.default_rng(42)
X = rng.uniform(0, np.pi, (60, 4))
dataset = Dataset(data=X)

## 1. `encoding_sensitivity()` — per-feature infidelity

In [ ]:
enc = qd.AngleEncoder()

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    sensitivity = qd.encoding_sensitivity(enc, dataset, epsilon=0.01, n_samples=20, seed=42)

print(f"Features : {len(sensitivity.feature_names)}")
print(f"Epsilon  : {sensitivity.epsilon}")
print(f"Samples  : {sensitivity.n_samples}")
print()
print(f"{'Feature':<10} {'Infidelity':>12}")
print("─" * 24)
for i, score in enumerate(sensitivity.scores):
    print(f"x{i:<9} {score:>12.6f}")

print(f"\nMost sensitive : {sensitivity.most_sensitive(2)}")

## 2. `score_encoding()` — expressibility, entanglement, kernel alignment

In [ ]:
encoders_to_score = [
    ("angle",       qd.AngleEncoder()),
    ("dense_angle", qd.DenseAngleEncoder()),
    ("entangled",   qd.EntangledAngleEncoder()),
]

for name, encoder in encoders_to_score:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", QuPrepWarning)
        scores = qd.score_encoding(encoder, dataset, n_samples=20, seed=42)
    def _fmt(v):
        return f"{v:.4f}" if v is not None else "N/A"
    expr = _fmt(scores.expressibility)
    ent  = _fmt(scores.entanglement_capability)
    ka   = _fmt(scores.kernel_alignment)
    print(f"{name:<15} expr={expr}  entanglement={ent}  kernel={ka}")

## 3. `detect_barren_plateau()` — gradient variance risk

In [ ]:
encoders_bp = [
    ("angle (shallow)",       qd.AngleEncoder(),                         dataset),
    ("IQP (deep, large)",
     qd.IQPEncoder(reps=3),
     Dataset(data=rng.uniform(0, np.pi, (30, 8)))),
    ("entangled (moderate)",  qd.EntangledAngleEncoder(layers=2),        dataset),
]

for name, encoder, ds in encoders_bp:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", QuPrepWarning)
        bp = qd.detect_barren_plateau(encoder, ds, cost_type="local")
    print(f"{name:<30} qubits={bp.n_qubits}  depth={bp.circuit_depth}  risk={bp.risk_level}")

print("\nRisk levels: none < mild < high < severe")